In [9]:
# Chapter 1 - Section 1: Configuration and dependencies
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

def resolve_repo_root(start: Path) -> Path:
    """Resolve repository root from notebook location."""
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Unable to resolve repository root")

NOTEBOOK_PATH = Path.cwd() / "03_consolidated_dataset_pipeline.ipynb"
if not NOTEBOOK_PATH.exists():
    NOTEBOOK_PATH = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()

ROOT = resolve_repo_root(NOTEBOOK_PATH.resolve())
SCRIPT_PATH = ROOT / "scripts/stage2_consolidated_dataset_pipeline.py"

spec = importlib.util.spec_from_file_location("stage2_pipeline", SCRIPT_PATH)
if spec is None or spec.loader is None:
    raise RuntimeError("Unable to load stage2 pipeline module")
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

print(json.dumps({"repo_root": str(ROOT), "pipeline_script": str(SCRIPT_PATH)}, indent=2))

{
  "repo_root": "/Users/mauriciorodriguez/Projects/AI-Masters/SAT-Dengue-V2",
  "pipeline_script": "/Users/mauriciorodriguez/Projects/AI-Masters/SAT-Dengue-V2/scripts/stage2_consolidated_dataset_pipeline.py"
}


In [10]:
# Chapter 1 - Section 2: Input and schema validation
paths = pipeline.build_paths(ROOT)
epi_columns = pipeline.validate_epi_headers(paths)
climate_columns = pipeline.validate_climate_header(paths)
catalog = pipeline.load_scope_catalog(paths)

summary = {
    "epi_column_count": len(epi_columns),
    "climate_column_count": len(climate_columns),
    "scope_municipality_count": int(catalog["municipality_code"].nunique()),
}
print(json.dumps(summary, indent=2))

{
  "epi_column_count": 69,
  "climate_column_count": 18,
  "scope_municipality_count": 76
}


In [11]:
# Chapter 1 - Section 3: Auxiliary normalization checks
sample = ["1", "001", "A2", ""]
normalized = [pipeline.normalize_numeric_code(value, 3) for value in sample]
print(json.dumps({"sample": sample, "normalized": normalized}, indent=2))

{
  "sample": [
    "1",
    "001",
    "A2",
    ""
  ],
  "normalized": [
    "001",
    "001",
    "",
    ""
  ]
}


In [12]:
# Chapter 1 - Section 4: Individual consolidation execution
config = pipeline.PipelineConfig(
    chunksize_epi=200_000,
    chunksize_climate=250_000,
    contract_version="candidate-construction-v1",
    perform_safe_write=True,
)
result = pipeline.run_pipeline(root=ROOT, config=config)
print(json.dumps({
    "consolidated_path": str(result.consolidated_path.relative_to(ROOT)),
    "panel_path": str(result.panel_path.relative_to(ROOT)),
    "consolidated_rows": result.consolidated_rows,
    "panel_rows": result.panel_rows,
}, indent=2))

{
  "consolidated_path": "data/processed/dengue_y_dengue_grave.csv",
  "panel_path": "data/processed/panel_mensual_dengue_clima.csv",
  "consolidated_rows": 1755705,
  "panel_rows": 17936
}


In [13]:
# Chapter 1 - Section 5: Daily climate join audit
manifest_path = ROOT / "data/interim/stage2_consolidated_dataset_pipeline/execution_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
join_audit = {
    "no_climate_match_assignable_rows": manifest["reconciliation"]["no_climate_match_assignable_rows"],
    "outside_scope_rows": manifest["reconciliation"]["outside_scope_rows"],
    "missing_or_unparseable_onset_rows": manifest["reconciliation"]["missing_or_unparseable_onset_rows"],
}
print(json.dumps(join_audit, indent=2))

{
  "no_climate_match_assignable_rows": 112,
  "outside_scope_rows": 1515860,
  "missing_or_unparseable_onset_rows": 136
}


In [14]:
# Chapter 1 - Section 6: Monthly panel inspection
panel_path = ROOT / "data/processed/panel_mensual_dengue_clima.csv"
panel = pipeline.pd.read_csv(panel_path)
panel_summary = {
    "rows": int(len(panel)),
    "columns": int(len(panel.columns)),
    "unique_municipalities": int(panel["municipality_code"].nunique()),
    "month_min": str(panel["year_month"].min()),
    "month_max": str(panel["year_month"].max()),
    "zero_in_coverage_rows": int((panel["epi_count_status"] == "zero_in_coverage").sum()),
    "outside_coverage_missing_rows": int((panel["epi_count_status"] == "outside_epi_coverage_missing").sum()),
}
for horizon in (1, 2):
    suffix = f"t{horizon}"
    evaluable_col = f"y_evaluable_{suffix}"
    status_col = f"y_status_{suffix}"
    label_col = f"y_{suffix}"
    if evaluable_col in panel.columns and status_col in panel.columns and label_col in panel.columns:
        panel_summary[f"{suffix}_evaluable_rows"] = int(panel[evaluable_col].fillna(False).sum())
        panel_summary[f"{suffix}_non_evaluable_rows"] = int((~panel[evaluable_col].fillna(False)).sum())
        panel_summary[f"{suffix}_status_counts"] = {
            str(key): int(value)
            for key, value in panel[status_col].value_counts(dropna=False).items()
        }
        panel_summary[f"{suffix}_label_counts"] = {
            ("NA" if pipeline.pd.isna(key) else str(int(key))): int(value)
            for key, value in panel[label_col].value_counts(dropna=False).items()
        }
print(json.dumps(panel_summary, indent=2))

{
  "rows": 17936,
  "columns": 47,
  "unique_municipalities": 76,
  "month_min": "2006-06",
  "month_max": "2026-01",
  "zero_in_coverage_rows": 6263,
  "outside_coverage_missing_rows": 0,
  "t1_evaluable_rows": 13376,
  "t1_non_evaluable_rows": 4560,
  "t1_status_counts": {
    "evaluable": 13376,
    "insufficient_history": 4484,
    "target_outside_panel": 76
  },
  "t1_label_counts": {
    "0": 8930,
    "NA": 4560,
    "1": 4446
  },
  "t2_evaluable_rows": 13376,
  "t2_non_evaluable_rows": 4560,
  "t2_status_counts": {
    "evaluable": 13376,
    "insufficient_history": 4408,
    "target_outside_panel": 152
  },
  "t2_label_counts": {
    "0": 8930,
    "NA": 4560,
    "1": 4446
  }
}


In [15]:
# Chapter 1 - Section 7: Reconciliation and validations
validator_path = ROOT / ".dwp/plans/PLAN_014_stage2_y_labels/validate_y_outputs.py"
validator_spec = importlib.util.spec_from_file_location("validator", validator_path)
if validator_spec is None or validator_spec.loader is None:
    raise RuntimeError("Unable to load validation module")
validator = importlib.util.module_from_spec(validator_spec)
sys.modules[validator_spec.name] = validator
validator_spec.loader.exec_module(validator)
validation_report = validator.validate_quick(ROOT, pipeline)
print(json.dumps(validation_report, indent=2))

{
  "consolidated_rows": 1755705,
  "panel_rows": 17936,
  "traceability_rows": 35872,
  "consolidated_sha256": "315a43aab561fa29a45c657c635c511b201847ca530eae0285e1522b2f7942ed",
  "panel_sha256": "907efacb8e044a400bd1aca4cb2b96f9e3fad6705df0e22fe320f25fb573b2fe",
  "traceability_sha256": "c99aad6cf87fb8b0dba46f9468b5ba2ff9c7a5957baea988109a39650339de03",
  "horizons": {
    "t1": {
      "evaluable_rows": 13376,
      "non_evaluable_rows": 4560,
      "positive_labels": 4446
    },
    "t2": {
      "evaluable_rows": 13376,
      "non_evaluable_rows": 4560,
      "positive_labels": 4446
    }
  }
}


In [16]:
# Chapter 1 - Section 8: Safe write outputs and execution summary
contract_path = ROOT / "data/interim/stage2_consolidated_dataset_pipeline/candidate_construction_contract_v1.json"
contract = json.loads(contract_path.read_text(encoding="utf-8"))
summary = {
    "candidate_contract_version": contract["contract_version"],
    "classification": contract["classification"],
    "y_labeling_policy": contract.get("target_labeling_policy", {}),
    "manifest_path": "data/interim/stage2_consolidated_dataset_pipeline/execution_manifest.json",
    "consolidated_csv": "data/processed/dengue_y_dengue_grave.csv",
    "panel_csv": "data/processed/panel_mensual_dengue_clima.csv",
    "y_traceability_csv": "data/interim/stage2_consolidated_dataset_pipeline/y_label_history_traceability.csv",
}
print(json.dumps(summary, indent=2))

{
  "candidate_contract_version": "candidate-construction-v1",
  "classification": "candidate_construction_contract",
  "y_labeling_policy": {
    "series": "dengue_notifications_count",
    "horizons_months": [
      1,
      2
    ],
    "reference_years": 5,
    "seasonality_basis": "same_calendar_month",
    "decision_rule": "label_1_if_observed_strictly_greater_than_p75",
    "non_evaluable_rule": "missing_future_or_insufficient_history_yields_null_label",
    "percentile": 0.75,
    "percentile_method": "hyndman_fan_type7_linear"
  },
  "manifest_path": "data/interim/stage2_consolidated_dataset_pipeline/execution_manifest.json",
  "consolidated_csv": "data/processed/dengue_y_dengue_grave.csv",
  "panel_csv": "data/processed/panel_mensual_dengue_clima.csv",
  "y_traceability_csv": "data/interim/stage2_consolidated_dataset_pipeline/y_label_history_traceability.csv"
}
